# ACTIVA M1 — Step 4 gate: raw MP4 → SlowFast features vs published features

**Settings:** Accelerator GPU T4, Internet ON.

For a few short EPIC-100 manifest videos this notebook
1. downloads the raw MP4s with the official EPIC downloader,
2. runs ACTIVA's `code/slowfast_features.py` (EPIC-100 SlowFast, `SlowFast.pyth`) in several preprocessing variants,
3. compares each variant with ActionFormer's **published** features (`feats`) and with the
   published per-window verb/noun probabilities (`v_prob`, `n_prob`) of the same model.

No ground-truth annotations are used: this only checks that our extractor reproduces the published inputs.
Output: `/kaggle/working/m1_step4_gate.json`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!which ffprobe ffmpeg
import sys, torch, numpy; print(sys.version.split()[0], "torch", torch.__version__, "cuda", torch.cuda.is_available(), "numpy", numpy.__version__)

## 1. Code: ACTIVA branch + epic-kitchens-slowfast (pinned)

In [ ]:
%cd /kaggle/working
!rm -rf ACTIVA eks dl
!git clone -q -b m1-real-action-inference https://github.com/harshjha15335/ACTIVA.git && git -C ACTIVA log --oneline -1
!git clone -q https://github.com/epic-kitchens/epic-kitchens-slowfast.git eks && git -C eks checkout -q bf50519 && git -C eks log --oneline -1
!git clone -q https://github.com/epic-kitchens/epic-kitchens-download-scripts.git dl && git -C dl log --oneline -1
!pip install -q fvcore iopath simplejson av gdown
import sys; sys.path[:0] = ["/kaggle/working/ACTIVA/code"]

## 2. Weights: SlowFast.pyth (EPIC-100) from the epic-kitchens-slowfast README

In [ ]:
import hashlib, pathlib, subprocess
CKPT = pathlib.Path("/tmp/SlowFast.pyth")
if not CKPT.exists():
    subprocess.run(["wget", "-q", "-O", str(CKPT), "https://www.dropbox.com/s/uxb6i2xkn91xqzi/SlowFast.pyth?dl=1"], check=True)
h = hashlib.sha256(CKPT.read_bytes()).hexdigest()
print(CKPT, CKPT.stat().st_size, "bytes  sha256", h)

## 3. Raw videos (official EPIC downloader)

In [ ]:
VIDEOS = ["P02_13", "P03_21", "P26_30", "P28_23"]   # short manifest videos, 4 participants
!cd /kaggle/working/dl && python epic_downloader.py --videos --specific-videos {",".join(VIDEOS)} --output-path /tmp/epic > /kaggle/working/dl.log 2>&1; tail -3 /kaggle/working/dl.log
import glob, json, subprocess
MP4 = {}
for v in VIDEOS:
    hits = glob.glob(f"/tmp/epic/**/{v}.MP4", recursive=True) + glob.glob(f"/tmp/epic/**/{v}.mp4", recursive=True)
    MP4[v] = hits[0] if hits else None
    if hits:
        p = json.loads(subprocess.run(["ffprobe","-v","error","-select_streams","v:0","-show_entries",
              "stream=width,height,avg_frame_rate,r_frame_rate,duration","-of","json",hits[0]],capture_output=True,text=True).stdout)["streams"][0]
        print(v, hits[0], p)
    else:
        print(v, "NOT DOWNLOADED")

## 4. Published features for the same videos (ActionFormer archive)

In [ ]:
import tarfile, numpy as np, os
TAR = "/tmp/epic_kitchens.tar.gz"
if not os.path.exists(TAR):
    subprocess.run(["gdown", "--fuzzy", "https://drive.google.com/file/d/1Z4U_dLuu6_cV5NBIrSzsSDOOj2Uar85X/view", "-O", TAR], check=True)
want = {f"{v}.npz" for v in VIDEOS}
PUB = {}
with tarfile.open(TAR, "r:gz") as t:
    for m in t:
        name = os.path.basename(m.name)
        if name in want:
            f = np.load(t.extractfile(m)); PUB[name[:-4]] = {k: f[k] for k in f.files}
            if len(PUB) == len(want): break
for v, d in PUB.items(): print(v, {k: x.shape for k, x in d.items()})

## 5. Load SlowFast and decode each video once (30 fps, 456x256 RGB)

In [ ]:
import time, slowfast_features as sf, video_io
from slowfast_features import Preprocess
bundle = sf.load_slowfast("/kaggle/working/eks", "/kaggle/working/eks/configs/EPIC-KITCHENS/SLOWFAST_8x8_R50.yaml", CKPT, device="cuda")
print("model loaded")
FRAMES, INFO = {}, {}
for v, path in MP4.items():
    if not path: continue
    t0 = time.time()
    INFO[v] = video_io.inspect_video(path)
    FRAMES[v] = [sf.resize_frame(rgb, Preprocess(color="rgb")) for _t, rgb in video_io.iter_frames(path, 30.0)]
    print(v, INFO[v].duration, "s", INFO[v].fps, "fps ->", len(FRAMES[v]), "frames", FRAMES[v][0].shape, f"{time.time()-t0:.1f}s")

## 6. Variants vs published

In [ ]:
def cos_rows(a, b):
    a = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-12); b = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-12)
    return (a * b).sum(1)

VARIANTS = {
    "bgr_full":    Preprocess(color="bgr"),
    "rgb_full":    Preprocess(color="rgb"),
    "bgr_crop256": Preprocess(color="bgr", crop=256),
    "bgr_crop224": Preprocess(color="bgr", crop=224),
}
REPORT = {}
for name, p in VARIANTS.items():
    for v, frames in FRAMES.items():
        if v not in PUB: continue
        t0 = time.time()
        # frames are already 456x256 RGB; resize_frame only applies crop + colour order here
        rgb_in = iter(frames)
        out = sf.extract_features(None, bundle, Preprocess(color=p.color, height=256, width=456, crop=p.crop, pad_last=True),
                                  batch_size=8, frames=rgb_in)
        pub = PUB[v]; T = min(len(out["feats"]), len(pub["feats"]))
        shifts = {}
        for s in range(-3, 4):
            a = out["feats"][max(0, s):T + min(0, s)]; b = pub["feats"][max(0, -s):T - max(0, s)]
            shifts[s] = float(cos_rows(a, b).mean()) if len(a) and len(a) == len(b) else None
        c = cos_rows(out["feats"][:T], pub["feats"][:T])
        rec = {
            "T_ours": int(len(out["feats"])), "T_pub": int(len(pub["feats"])),
            "feat_cos_mean": float(c.mean()), "feat_cos_min": float(c.min()),
            "feat_cos_by_shift": shifts,
            "verb_top1_agree": float((out["v_prob"][:T].argmax(1) == pub["v_prob"][:T].argmax(1)).mean()),
            "noun_top1_agree": float((out["n_prob"][:T].argmax(1) == pub["n_prob"][:T].argmax(1)).mean()),
            "v_prob_mae": float(np.abs(out["v_prob"][:T] - pub["v_prob"][:T]).mean()),
            "n_prob_mae": float(np.abs(out["n_prob"][:T] - pub["n_prob"][:T]).mean()),
            "seconds": round(time.time() - t0, 2),
        }
        REPORT.setdefault(name, {})[v] = rec
        print(f"{name:12s} {v}: T {rec['T_ours']}/{rec['T_pub']}  cos mean {rec['feat_cos_mean']:.4f} min {rec['feat_cos_min']:.4f}  "
              f"verb top1 {rec['verb_top1_agree']:.2f} noun top1 {rec['noun_top1_agree']:.2f}  best shift {max(shifts, key=lambda k: shifts[k] or -9)}")

In [ ]:
summary = {name: {k: float(np.mean([r[k] for r in vids.values()])) for k in
                  ("feat_cos_mean", "feat_cos_min", "verb_top1_agree", "noun_top1_agree", "v_prob_mae", "n_prob_mae")}
           for name, vids in REPORT.items()}
for name, s in sorted(summary.items(), key=lambda kv: -kv[1]["feat_cos_mean"]):
    print(f"{name:12s}", {k: round(x, 4) for k, x in s.items()})
meta = {"checkpoint_sha256": h, "activa_commit": subprocess.run(["git","-C","/kaggle/working/ACTIVA","rev-parse","HEAD"],capture_output=True,text=True).stdout.strip(),
        "eks_commit": "bf50519", "torch": torch.__version__, "gpu": torch.cuda.get_device_name(0),
        "videos": {v: {"duration": INFO[v].duration, "fps": INFO[v].fps, "w": INFO[v].width, "h": INFO[v].height, "frames_30fps": len(FRAMES[v])} for v in FRAMES}}
json.dump({"meta": meta, "summary": summary, "per_video": REPORT}, open("/kaggle/working/m1_step4_gate.json", "w"), indent=1)
print("saved /kaggle/working/m1_step4_gate.json")

## Reading the result
PASS (team rule from the M1 guide): window counts match, mean cosine ≥ 0.95, and the model's verb/noun top-1 mostly agrees
with the published `v_prob`/`n_prob`. The best variant becomes the production default in `slowfast_features.Preprocess`.